# CDC deep dive: how the deduplication works, and what it buys

Ancestree deduplicates artifacts at two layers:

**Layer 1 — content-defined chunking (FastCDC).** Files are split into
variable-length chunks at boundaries chosen by the bytes themselves, via a
Gear rolling hash. Insert or delete a region and only the chunks around the
edit change — everything else keeps its bytes, keeps its SHA-256, and is
stored once. Chunks average 32 KiB (min 8, max 256), and files above 64 MB
skip the rolling hash for fixed boundaries so huge ingests stay at C speed.

**Layer 2 — resemblance + deltas.** Exact matching misses chunks that are
*almost* identical (a config value tweaked, floats re-encoded). Each new
chunk gets four cheap "super-features" (min-wise hashes over strided
samples); sharing any feature with a stored chunk nominates that chunk as a
base, and the newcomer is stored as a **delta**: zlib compression with the
base as its preset dictionary. DEFLATE's back-references into the base ARE
the delta — C speed, standard library. A delta is only kept when it actually
beats plain compression, its base is always a raw chunk (so a read is never
more than two fetches), and the whole layer sits behind the store's `chunk`
policy.

This notebook shows both layers working, then measures **save time, load
time and storage** across different kinds of file, with Layer 2 on and off.

In [1]:
import random
import shutil
import string
import tempfile
import time
from pathlib import Path

from ancestree import LineageStore
from ancestree.ingest.cdc import chunk_bytes

workdir = Path(tempfile.mkdtemp(prefix="ancestree-cdc-"))

def human(n):
    for unit in ("B", "KiB", "MiB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} GiB" 

## Layer 1 mechanics: boundaries that survive an insertion

Chunk a megabyte of random bytes, then insert 100 bytes into the middle and
chunk again. Fixed-size chunking would shift every boundary after the edit;
content-defined boundaries re-align immediately.

In [2]:
data = random.Random(1).randbytes(1_000_000)
chunks = list(chunk_bytes(data))
sizes = sorted(len(c) for c in chunks)
print(f"{len(chunks)} chunks | min {human(sizes[0])} | median {human(sizes[len(sizes)//2])} | max {human(sizes[-1])}")

edited = data[:500_000] + random.Random(2).randbytes(100) + data[500_000:]
before, after = set(chunk_bytes(data)), set(chunk_bytes(edited))
shared = len(before & after) / len(before)
print(f"after inserting 100 bytes mid-file: {shared:.0%} of chunks are byte-identical")
assert shared > 0.9

28 chunks | min 6.4 KiB | median 35.7 KiB | max 49.9 KiB


after inserting 100 bytes mid-file: 96% of chunks are byte-identical


## Layer 2 mechanics: near-duplicates become deltas

Two nodes whose single-chunk artifacts differ by 40 scattered bytes. With the
`chunk` policy on (the default), the second lands as a `kind=1` delta against
the first — visible straight from the schema with `store.sql`.

In [3]:
def mutate(data, edits, seed):
    rng = random.Random(seed)
    out = bytearray(data)
    for _ in range(edits):
        out[rng.randrange(len(out))] = rng.randrange(256)
    return bytes(out)

demo = LineageStore(workdir / "demo")
base_bytes = random.Random(3).randbytes(30_000)
for version, payload in enumerate([base_bytes, mutate(base_bytes, 40, seed=4)]):
    with demo.create_node(step_type="version") as node:
        (node / "state.bin").write_bytes(payload)
        node.add_meta("v", version)

for row in demo.sql(
    "SELECT kind, count(*) AS n, sum(length(data)) AS stored, sum(length) AS plain "
    "FROM chunk GROUP BY kind"
):
    label = "raw " if row["kind"] == 0 else "delta"
    print(f"kind={label} chunks={row['n']} plain={human(row['plain'])} stored={human(row['stored'])}")
demo.close()

kind=raw  chunks=1 plain=29.3 KiB stored=29.3 KiB
kind=delta chunks=1 plain=29.3 KiB stored=408.0 B


## The measurement rig

Four kinds of file, chosen to behave differently:

- **random binary** — incompressible; versions differ by ~1% scattered edits.
  Exact chunk matching shares nothing here; this is Layer 2's home turf.
- **csv text** — repetitive and compressible; versions edit some values and
  append rows. Both layers get something.
- **json config** — structured text, a handful of values change per version.
- **identical copies** — the same bytes saved repeatedly; pure Layer 1.

Each type saves 8 versions (~256 KiB each) into two fresh stores — Layer 1
only (`chunk=False`) and Layer 1+2 (`chunk=True`) — timing every save, then
timing a cold read-back of every version and comparing stored bytes.

In [4]:
SIZE = 256 * 1024
VERSIONS = 8

def make_versions(kind):
    rng = random.Random(10)
    versions = []
    if kind == "random binary":
        payload = rng.randbytes(SIZE)
        for v in range(VERSIONS):
            payload = mutate(payload, SIZE // 100, seed=20 + v)
            versions.append(payload)
    elif kind == "csv text":
        rows = [f"{i},{rng.random():.6f},{''.join(rng.choices(string.ascii_lowercase, k=8))}"
                for i in range(4_000)]
        for v in range(VERSIONS):
            for _ in range(40):  # edit some values
                i = rng.randrange(len(rows))
                rows[i] = f"{i},{rng.random():.6f},{rows[i].rsplit(',', 1)[1]}"
            rows.extend(f"{len(rows)+j},{rng.random():.6f},appended" for j in range(50))
            versions.append("\n".join(rows).encode())
    elif kind == "json config":
        import json as _json
        config = {f"param_{i}": rng.random() for i in range(6_000)}
        for v in range(VERSIONS):
            for key in rng.sample(list(config), 30):
                config[key] = rng.random()
            versions.append(_json.dumps(config, indent=1).encode())
    elif kind == "identical copies":
        payload = rng.randbytes(SIZE)
        versions = [payload] * VERSIONS
    return versions

def measure(kind, chunk_policy):
    root = workdir / f"{kind.replace(' ', '_')}-{chunk_policy}"
    store = LineageStore(root, chunk=chunk_policy)
    versions = make_versions(kind)

    started = time.perf_counter()
    for v, payload in enumerate(versions):
        with store.create_node(step_type="version") as node:
            (node / "data.bin").write_bytes(payload)
            node.add_meta("v", v)  # identical copies still make distinct nodes
    save_s = time.perf_counter() - started

    started = time.perf_counter()
    total = 0
    for record in store.find():
        total += len((record / "data.bin").read_bytes())
    load_s = time.perf_counter() - started

    stats = store.stats()
    store.close()
    return {
        "kind": kind,
        "policy": "L1+2" if chunk_policy else "L1  ",
        "save_s": save_s,
        "load_s": load_s,
        "logical": stats["logical_bytes"],
        "stored": stats["chunk_stored_bytes"],
        "ratio": stats["dedup_ratio"],
    }

results = []
for kind in ("random binary", "csv text", "json config", "identical copies"):
    for policy in (False, True):
        results.append(measure(kind, policy))
print("measured", len(results), "runs")

measured 8 runs


## Results

In [5]:
header = f"{'file type':<18} {'policy':<6} {'save_s':>7} {'load_s':>7} {'logical':>10} {'stored':>10} {'saving':>7} {'ratio':>6}"
print(header)
print("-" * len(header))
for r in results:
    saving = 1 - r["stored"] / r["logical"]
    print(f"{r['kind']:<18} {r['policy']:<6} {r['save_s']:>7.3f} {r['load_s']:>7.3f} "
          f"{human(r['logical']):>10} {human(r['stored']):>10} {saving:>6.0%} {str(r['ratio']):>6}")

by = {(r["kind"], r["policy"].strip()): r for r in results}
for kind in ("random binary", "csv text", "json config", "identical copies"):
    l1, l2 = by[(kind, "L1")], by[(kind, "L1+2")]
    print(f"{kind:<18} Layer 2 stores {l1['stored'] / max(l2['stored'], 1):.1f}x less than Layer 1 alone")

file type          policy  save_s  load_s    logical     stored  saving  ratio
------------------------------------------------------------------------------
random binary      L1       0.492   0.007    2.0 MiB    2.0 MiB    -0%  0.999
random binary      L1+2     0.512   0.009    2.0 MiB  894.7 KiB    56%  2.289
csv text           L1       0.320   0.006  750.5 KiB  418.2 KiB    44%  1.795
csv text           L1+2     0.340   0.007  750.5 KiB  130.3 KiB    83%  5.761
json config        L1       0.429   0.008    1.6 MiB  572.9 KiB    65%  2.869
json config        L1+2     0.463   0.009    1.6 MiB  326.8 KiB    80%  5.029
identical copies   L1       0.436   0.006    2.0 MiB  256.1 KiB    87%  7.996
identical copies   L1+2     0.434   0.006    2.0 MiB  256.1 KiB    87%  7.996
random binary      Layer 2 stores 2.3x less than Layer 1 alone
csv text           Layer 2 stores 3.2x less than Layer 1 alone
json config        Layer 2 stores 1.8x less than Layer 1 alone
identical copies   Layer 2 st

## Reading the numbers

- **Identical copies** are free at either layer: the second copy of every
  chunk already exists, so eight versions cost one. (Node-level dedup would
  normally collapse these into one node outright — the per-version metadata
  here keeps them distinct so the chunk layer is what's being measured.)
- **Random binary with scattered edits** is where Layer 2 earns its keep:
  exact matching shares nothing (every chunk differs somewhere), while the
  delta layer stores each new version as small patches against the last.
- **Text formats** compress well on their own, so the absolute numbers are
  smaller, but deltas still beat freshly-compressed chunks wherever a
  similar chunk already exists.
- Save time with Layer 2 on carries a modest premium (feature lookups plus a
  trial encode per new chunk); load time carries the delta-decode. Both stay
  in the same order of magnitude — and if a store's read path matters more
  than its size, `chunk=False` at creation turns Layer 2 off for good.

The chunking constants (8/32/256 KiB, the 64 MB fixed-boundary fallback) and
the full reasoning live in REBUILD_BLUEPRINT.md; the repeatable benchmark
behind the headline number is `benchmarks/layer2.py`.

In [6]:
shutil.rmtree(workdir)
print("cleaned up")

cleaned up
